# Fine-Tuning DistilBERT for Emotion Classification

This notebook demonstrates a complete NLP workflow using the Hugging Face `transformers` library. We will load an emotion dataset, fine-tune a pre-trained DistilBERT model, and perform inference.

In [5]:
!pip install transformers datasets evaluate -q

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


## 1. Data Loading
We use the `datasets` library to load the 'emotion' dataset, which contains Twitter messages labeled with one of six emotions: sadness, joy, love, anger, fear, and surprise.

In [6]:
dataset = load_dataset("emotion")
print(dataset)
print(dataset["train"][0])
print(dataset["train"].features)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': 'i didnt feel humiliated', 'label': 0}
{'text': Value('string'), 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}


## 2. Tokenization
Raw text must be converted into numerical tokens. We use `AutoTokenizer` to load the DistilBERT tokenizer and apply padding and truncation to ensure consistent input lengths.

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## 3. Model Initialization
We load `distilbert-base-uncased` with a sequence classification head. Since our dataset has 6 emotion categories, we set `num_labels=6`.

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels = 6
    )

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 4. Metrics and Evaluation
To monitor the model's performance during training, we define a function to calculate Accuracy and the Weighted F1-score.

In [9]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    }

## 5. Fine-Tuning
Using the `Trainer` API, we configure the training arguments (learning rate, epochs, etc.) and run the fine-tuning process. We include a `DataCollatorWithPadding` to handle dynamic padding during batching.

In [10]:
from transformers import DataCollatorWithPadding

# Initialize the data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,  # Added this to fix the dimension error
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.271859,0.218812,0.919500,0.920356
2,0.147551,0.165982,0.934500,0.934531


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.39581287622451783, metrics={'train_runtime': 291.08, 'train_samples_per_second': 109.935, 'train_steps_per_second': 3.435, 'total_flos': 709637076406656.0, 'train_loss': 0.39581287622451783, 'epoch': 2.0})

In [11]:
from transformers import pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)
print(classifier("I cannot believe I finally finished this project!"))
print(classifier("I'm so worried about the interview tomorrow."))

[{'label': 'LABEL_1', 'score': 0.8803591728210449}]
[{'label': 'LABEL_4', 'score': 0.8925203680992126}]


In [10]:
print(classifier("I cant wait to get a job"))

[{'label': 'LABEL_1', 'score': 0.5176132917404175}]


## 6. Inference and Label Mapping
Initially, the model outputs generic labels like `LABEL_1`. We map these IDs to their actual emotion names (e.g., 'joy', 'fear') and use the `pipeline` API for easy testing.

In [2]:
id2label = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
label2id = {v: k for k, v in id2label.items()}


In [12]:
model.config.id2label = id2label
model.config.label2id = label2id

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)
print(classifier("I'm so worried about the interview tomorrow."))
# [{'label': 'fear', 'score': 0.942}]

[{'label': 'fear', 'score': 0.8925203680992126}]


In [13]:
print(classifier("I cant find the key"))

[{'label': 'joy', 'score': 0.42316263914108276}]


The model struggles with neutral or ambiguous statements (e.g., 'I can't find the key' → joy at 0.42 confidence). It was trained on emotionally expressive tweets, so out-of-distribution inputs produce low-confidence predictions. A production version would use a confidence threshold or include a 'neutral' class.


In [14]:
print(classifier("I'm so frustrated I can't find my keys anywhere!"))

[{'label': 'anger', 'score': 0.9843594431877136}]
